![image_1780906756442.png](./image_1780906756442.png "image_1780906756442.png")

![image_1780906788320.png](./image_1780906788320.png "image_1780906788320.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("MeasurementsDF").getOrCreate()

# Create dataset
data = [
    (1, 2, "2022-07-10 08:00:00"),
    (2, 5, "2022-07-10 09:00:00"),
    (3, 3, "2022-07-10 11:00:00"),
    (4, 7, "2022-07-11 07:00:00"),
    (5, 1.5, "2022-07-11 10:00:00"),
    (6, 4, "2022-07-11 14:00:00"),
    (7, 6, "2022-07-11 16:00:00"),
]

# Define explicit schema with DoubleType for measurement_value
schema = StructType([
    StructField("measurement_id", IntegerType(), True),
    StructField("measurement_value", DoubleType(), True),
    StructField("measurement_time", StringType(), True)
])

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Convert measurement_time to timestamp type
df = df.withColumn("measurement_time", f.to_timestamp("measurement_time"))

# Show DataFrame
df.show()

In [0]:
result_df = (
    df.withColumn("measurement_day", f.to_date(f.col("measurement_time")))
    .withColumn(
        "rn",
        f.row_number().over(
            Window.partitionBy("measurement_day").orderBy(f.col("measurement_time"))
        ),
    )
    .select(
        f.col("measurement_day"),
        f.sum(
            f.when(f.col("rn") % 2 != 0, f.col("measurement_value")).otherwise(f.lit(0))
        )
        .over(Window.partitionBy("measurement_day"))
        .alias("odd_sum"),
        f.sum(
            f.when(f.col("rn") % 2 == 0, f.col("measurement_value")).otherwise(f.lit(0))
        )
        .over(Window.partitionBy("measurement_day"))
        .alias("even_sum"),
    )
    .distinct()
    .orderBy(f.col("measurement_day"))
)
display(result_df)